In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')

np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'bpic_2017'

n_processes = 96
batch_size = 50

log_name = 'test'

with open('../transformed_event_logs/BPIC_2017_all_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_resources = ['User_1','User_10','User_100','User_101','User_102','User_103','User_104','User_105','User_106','User_107','User_108','User_109','User_11','User_110','User_111','User_112','User_113','User_114','User_115','User_116','User_117','User_118','User_119','User_12','User_120','User_121','User_122','User_123','User_124','User_125','User_126','User_127','User_128','User_129','User_13','User_130','User_131','User_132','User_133','User_134','User_135','User_136','User_137','User_138','User_139','User_14','User_140','User_141','User_142','User_143','User_144','User_145','User_146','User_147','User_148','User_149','User_15','User_16','User_17','User_18','User_19','User_2','User_20','User_21','User_22','User_23','User_24','User_25','User_26','User_27','User_28','User_29','User_3','User_30','User_31','User_32','User_33','User_34','User_35','User_36','User_37','User_38','User_39','User_4','User_40','User_41','User_42','User_43','User_44','User_45','User_46','User_47','User_48','User_49','User_5','User_50','User_51','User_52','User_53','User_54','User_55','User_56','User_57','User_58','User_59','User_6','User_60','User_61','User_62','User_63','User_64','User_65','User_66','User_67','User_68','User_69','User_7','User_70','User_71','User_72','User_73','User_74','User_75','User_76','User_77','User_78','User_79','User_8','User_80','User_81','User_82','User_83','User_84','User_85','User_86','User_87','User_88','User_89','User_9','User_90','User_91','User_92','User_93','User_94','User_95','User_96','User_97','User_98','User_99']
known_activities = ['W_Assess potential fraud__ate_abort','W_Assess potential fraud__complete','W_Assess potential fraud__resume','W_Assess potential fraud__schedule','W_Assess potential fraud__start','W_Assess potential fraud__suspend','W_Assess potential fraud__withdraw','W_Call after offers__ate_abort','W_Call after offers__complete','W_Call after offers__resume','W_Call after offers__schedule','W_Call after offers__start','W_Call after offers__suspend','W_Call after offers__withdraw','W_Call incomplete files__ate_abort','W_Call incomplete files__complete','W_Call incomplete files__resume','W_Call incomplete files__schedule','W_Call incomplete files__start','W_Call incomplete files__suspend','W_Complete application__ate_abort','W_Complete application__complete','W_Complete application__resume','W_Complete application__schedule','W_Complete application__start','W_Complete application__suspend','W_Handle leads__complete','W_Handle leads__resume','W_Handle leads__schedule','W_Handle leads__start','W_Handle leads__suspend','W_Handle leads__withdraw','W_Shortened completion __resume','W_Shortened completion __schedule','W_Shortened completion __start','W_Shortened completion __suspend','W_Validate application__ate_abort','W_Validate application__complete','W_Validate application__resume','W_Validate application__schedule','W_Validate application__start','W_Validate application__suspend']

In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : '',
                                                        'resources' : False,
                                                        'categorical_args' : ['concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

In [4]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-3.938891675440818214958673679')

In [5]:
np.mean(get_pscores(likelihoods_A))

np.float64(771027.7205473868)

In [6]:
drbart_model_path = '../../../models/advanced/'+model_name+'/resource/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'resources' : False,
                                                        'categorical_args' : ['resource'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.078748557543250678242230737')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(745349.7852130482)

In [9]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['resource', 'concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!


In [10]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-3.890875821730397904895582344')

In [11]:
np.mean(get_pscores(likelihoods_A))

np.float64(751680.6576919636)

In [12]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['resource', 'concept_name'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

sampled duration is still above limit!


In [13]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-3.908692602331371663594558329')

In [14]:
np.mean(get_pscores(likelihoods_A))

np.float64(775750.3998037613)

In [15]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!


In [16]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.014486934082825892327143389')

In [17]:
np.mean(get_pscores(likelihoods_A))

np.float64(857677.7527187085)

In [18]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource_count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

sampled duration is still above limit!


In [19]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-6.486427993464760074003136554')

In [20]:
np.mean(get_pscores(likelihoods_A))

np.float64(938479.1918171725)

In [21]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

In [22]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.521340845571665706925631519')

In [23]:
np.mean(get_pscores(likelihoods_A))

np.float64(764219.6716978532)

In [24]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_resource-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

In [25]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-3.956850264077427926678123870')

In [26]:
np.mean(get_pscores(likelihoods_A))

np.float64(796206.4951012752)

In [27]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week_activity-count_resoure-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still above limit!
sampled duration is still

In [28]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4302.088859626559110449948613')

In [29]:
np.mean(get_pscores(likelihoods_A))

np.float64(1197387.8536767939)